# Optimizer Agent Notebook

This notebook demonstrates the Optimizer Agent's capabilities in improving quantum circuits.

## Purpose
The Optimizer Agent refines generated circuits to improve performance and efficiency. This notebook covers:

1.  **Heuristic Optimization**: Applying standard Braket optimizations like gate merging and dropping negligible operations.
2.  **RL Optimization**: Using a Reinforcement Learning loop to iteratively improve circuit metrics (depth, gate count) based on a reward function.
3.  **Comparisoon**: Analyzing and comparing the metrics of the original vs. optimized circuits.

## Usage
Run this notebook to optimize existing Braket circuits and observe the improvements in circuit depth and gate count.


In [1]:
import sys
import os
from pathlib import Path
from braket.circuits import Circuit

# Add project root to path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.braket_rag_code_assistant.config import get_config, setup_logging
from src.agents.optimizer import OptimizerAgent

# Setup logging
setup_logging()

2025-12-07 13:44:58 | INFO     | src.braket_rag_code_assistant.config.logging:setup_all:138 | Logging configuration completed


### Initialize Agent
Initialize the Optimizer Agent.

In [2]:
# First, add these imports and setup the RAG components
from src.rag.knowledge_base import KnowledgeBase
from src.rag.retriever import Retriever

# Initialize Knowledge Base with all datasets
KNOWLEDGE_BASE_DIR = project_root / "data" / "knowledge_base"
kb = KnowledgeBase(knowledge_base_path=KNOWLEDGE_BASE_DIR)
kb.load_from_directory()
kb.index_entries()

# Create retriever
retriever = Retriever(knowledge_base=kb)

# Initialize optimizer with RAG retriever
optimizer = OptimizerAgent(retriever=retriever)
print("Optimizer Agent initialized with RAG support.")

2025-12-07 13:44:58 | INFO     | config.config_loader:load:93 | ✅ Loaded configuration from D:\University\Uni\Semester 7\Generative AI\Project\Braket-RAG-Code-Assistant\config\config.json
2025-12-07 13:44:58 | INFO     | src.rag.embeddings:__init__:97 | Loading embedding model: BAAI/bge-base-en-v1.5
2025-12-07 13:44:58 | INFO     | src.rag.embeddings:__init__:98 | Using device: cpu


2025-12-07 13:44:58,820 - sentence_transformers.SentenceTransformer - INFO - SentenceTransformer.py:227 - Load pretrained SentenceTransformer: BAAI/bge-base-en-v1.5


2025-12-07 13:45:03 | INFO     | src.rag.embeddings:__init__:106 | ✅ Embedding model loaded successfully
2025-12-07 13:45:03 | INFO     | src.rag.embeddings:__init__:113 | Embedding dimension: 768
2025-12-07 13:45:03 | INFO     | src.rag.vector_store:_init_faiss:139 | Initialized FAISS index
2025-12-07 13:45:03 | INFO     | src.rag.vector_store:__init__:120 | Initialized VectorStore with faiss backend
2025-12-07 13:45:03 | INFO     | src.rag.vector_store:__init__:121 | Embedding dimension: 768
2025-12-07 13:45:03 | INFO     | src.rag.knowledge_base:__init__:100 | Initialized KnowledgeBase with 0 entries
2025-12-07 13:45:03 | INFO     | src.rag.knowledge_base:load_from_jsonl:116 | Loading knowledge base from D:\University\Uni\Semester 7\Generative AI\Project\Braket-RAG-Code-Assistant\data\knowledge_base\curated_designer_examples.jsonl
2025-12-07 13:45:03 | INFO     | src.rag.knowledge_base:load_from_jsonl:129 | Loaded 102 entries from D:\University\Uni\Semester 7\Generative AI\Project\B

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

2025-12-07 13:45:21 | INFO     | src.rag.knowledge_base:index_entries:248 | ✅ Indexed 135 entries
2025-12-07 13:45:21 | INFO     | src.rag.retriever:__init__:170 | Initialized Retriever with top_k=5, threshold=0.7, topic_boost=0.15, hybrid_scoring=True
2025-12-07 13:45:21 | INFO     | src.agents.base_agent:__init__:78 | Initialized OptimizerAgent agent
2025-12-07 13:45:21 | INFO     | src.rag.generator:__init__:170 | Initialized Generator with ollama/braket-optimizer-agent
2025-12-07 13:45:21 | INFO     | src.agents.optimizer:__init__:74 | OptimizerAgent initialized (RAG: enabled)


Optimizer Agent initialized with RAG support.


### Create Sample Circuit
Let's create an unoptimized circuit to test.

In [3]:
# Create a circuit with redundant gates
q0, q1 = braket.LineQubit.range(2)
circuit = braket.Circuit(
    braket.X(q0),
    braket.X(q0),  # Redundant (Identity)
    braket.Z(q1),
    braket.Z(q1),  # Redundant (Identity)
    braket.CNOT(q0, q1),
    braket.CNOT(q0, q1), # Redundant (Identity)
    braket.H(q0),
    braket.H(q0)   # Redundant (Identity)
)

print("Original Circuit:")
print(circuit)

Original Circuit:
0: ───X───X───@───@───H───H───
              │   │
1: ───Z───Z───X───X───────────


### Optimize Circuit
Now let's run the optimizer.

In [4]:
task = {
    "circuit": circuit,
    "optimization_level": "aggressive",
    "use_rl": True  # Enable RL optimization
}

try:
    result = optimizer.execute(task)
    
    if result['success']:
        print("Successfully optimized circuit!")
        print("\nOptimized Circuit:")
        print("-" * 40)
        print(result['optimized_code'])
        print("-" * 40)
        
        print("\nMetrics Comparison:")
        print(f"Original Depth: {result['original_metrics'].get('depth')}")
        print(f"Optimized Depth: {result['optimized_metrics'].get('depth')}")
        print(f"Original Gate Count: {result['original_metrics'].get('num_operations')}")
        print(f"Optimized Gate Count: {result['optimized_metrics'].get('num_operations')}")
        
        print("\nImprovements:")
        for imp in result['improvements']:
            print(f"- {imp}")
            
    else:
        print(f"Failed to optimize circuit: {result.get('error')}")
        
except Exception as e:
    print(f"Error executing task: {e}")

2025-12-07 13:45:21 | INFO     | src.agents.optimizer:execute:280 | Retrieved 3 optimization references from RAG
2025-12-07 13:45:21 | INFO     | src.agents.optimizer:_optimize_with_rl:420 | Starting RL optimization (iterations=10)


Successfully optimized circuit!

Optimized Circuit:
----------------------------------------
from braket.circuits import Circuit

qubits = braket.LineQubit.range(2)

circuit = braket.Circuit()
circuit.append(braket.CNOT(qubits[0], qubits[1]))
circuit.append(braket.CNOT(qubits[0], qubits[1]))

print(circuit)
----------------------------------------

Metrics Comparison:
Original Depth: 6
Optimized Depth: 2
Original Gate Count: 8
Optimized Gate Count: 2

Improvements:
- Circuit 2 has lower depth


### LLM Optimization
Use the LLM backend to optimize the circuit.

In [5]:
llm_task = {
    "circuit": circuit,
    "use_llm": True,
    "use_heuristics": False,
    "use_rl": False
}

try:
    print("Running LLM Optimization...")
    llm_result = optimizer.execute(llm_task)
    
    if llm_result['success']:
        print("Successfully optimized circuit using LLM!")
        print("\nLLM Optimized Circuit:")
        print("-" * 40)
        print(llm_result['optimized_code'])
        print("-" * 40)
        
        print("\nLLM Metrics:")
        print(f"Depth: {llm_result['optimized_metrics'].get('depth')}")
        print(f"Gate Count: {llm_result['optimized_metrics'].get('num_operations')}")
    else:
        print(f"LLM optimization failed: {llm_result.get('error')}")

except Exception as e:
    print(f"Error executing LLM task: {e}")

2025-12-07 13:45:21 | INFO     | src.agents.optimizer:execute:280 | Retrieved 3 optimization references from RAG
2025-12-07 13:45:21 | INFO     | src.agents.optimizer:execute:288 | Running LLM optimization
2025-12-07 13:45:21 | INFO     | src.rag.generator:generate_direct:483 | Generating code directly (no RAG) using ollama/braket-optimizer-agent


Running LLM Optimization...


2025-12-07 13:45:32 | INFO     | src.rag.generator:generate_direct:543 | ✅ Direct code generation completed


Successfully optimized circuit using LLM!

LLM Optimized Circuit:
----------------------------------------
from braket.circuits import Circuit

qubits = braket.LineQubit.range(2)

circuit = braket.Circuit()
# Apply X gate twice on qubit 0, which cancels out
# circuit.append(braket.X(qubits[0]))
# circuit.append(braket.X(qubits[0]))

# Apply Z gate twice on qubit 1, which also cancels out
# circuit.append(braket.Z(qubits[1]))
# circuit.append(braket.Z(qubits[1]))

# Apply CNOT gate twice, which is equivalent to no operation if both controls are the same
# circuit.append(braket.CNOT(qubits[0], qubits[1]))
# circuit.append(braket.CNOT(qubits[0], qubits[1]))

# Apply H gate twice on qubit 0, which reduces to a single Z gate
circuit.append(braket.Z(qubits[0]))

print(circuit)
----------------------------------------

LLM Metrics:
Depth: 1
Gate Count: 1
